In [6]:
# 데이터 로드
import pandas as pd

df = pd.read_parquet(r"C:\Users\Playdata\Desktop\SKN29-2nd-1Team\data\raw\youtube_trends_KR.parquet")
# df = pd.read_parquet(r"C:\skn29\test\data\youtube_trends_KR (1).parquet")
#df = pd.read_parquet(r"/Users/kyoungchan/skn29/python/datasets/youtube_trends_KR.parquet")

df.head(5)

,collection_date,region_code,rank,video_id,title,description,published_at,channel_id,channel_title,category_id,default_language,default_audio_language,live_broadcast_content,view_count,comment_count
0,2022-07-01 00:00:00,KR,1,k1mccn55MyA,[ENG][차이나는 밤샘토크] 술? 못해요🤣 장꾸미 폭발한 손석구와 함께한 '나의 ...,💡 배우｜손석구드라마에선 주당인 줄 알았는데..🍺실제로는 술을 잘 안 먹는다고?!반...,2022-06-29 13:00:09.000,UCe8HrlhMkYlQ_61EjrrQlHw,차이나는 클라스,24,<NA>,ko,,766265,1567.0
1,2022-07-01 00:00:00,KR,2,tz5ZHaa-qwA,2022 임영웅 전국투어 콘서트 ‘IM HERO’ Tour Spot (Live Ver.),#임영웅 #LimYoungWoong임영웅 웅튜브 구독♡좋아요♡알람설정열심히 할게요/...,2022-06-29 04:00:31.000,UC3WZlO2Zl8NE1yIUgtwUtQw,임영웅,10,ko,ko,,780914,4641.0
2,2022-07-01 00:00:00,KR,3,cfzpznYQ-Tg,[찬또야 어디가? 1화] 첫 힐링 여행지 양평에 찬또가 떴다?🌞,찬또야 어디가? 첫 여행지로 선택된 경기도 양평🚗첫 목적지는 찬또가 선택한 양평오일...,2022-06-30 16:00:31.000,UC4UnP3v-iaFaLdtKwp84Pmw,이찬원,10,<NA>,<NA>,,139139,2668.0
3,2022-07-01 00:00:00,KR,4,YlNe0TbblXY,"한국에서 맛보는 세계 - 중동 음식 with 알파고 시나씨, 공혁준",유부남은 못말려--------------------------------------...,2022-06-30 04:00:20.000,UCNhofiqfw5nl-NeDJkXtPvw,빠니보틀 Pani Bottle,19,ko,ko,,283863,1105.0
4,2022-07-01 00:00:00,KR,5,vJvX9L6FCWI,선미 (SUNMI) - '열이올라요 (Heart Burn)' Music Video,#선미 #SUNMI #열이올라요 #HeartBurn #❤️‍🔥⬆️💿 선미(SUNMI...,2022-06-29 14:00:11.000,UCsVpgRB8YHLWA0ZrkhtHvTA,선미 SUNMI,10,en,ko,,14111694,6303.0


In [7]:
# step1. 데이터 무결성 확인 + 한국 크리에이터 필터

# 1. datetime 변환 
# UTC → KST 변환 (영국 표준시 → 한국시간 +9h)
df['collection_date'] = (
    pd.to_datetime(df['collection_date'], utc=True)
    .dt.tz_convert('Asia/Seoul').
    dt.tz_localize(None))
df['published_at'] = (
    pd.to_datetime(df['published_at'], utc=True)
    .dt.tz_convert('Asia/Seoul')
    .dt.tz_localize(None)
)

# 2. 중복 제거
df = df.drop_duplicates(subset=['video_id', 'collection_date'])
print("중복 제거 후:", df.shape)

# 3. 한국 크리에이터 필터
korean_title_mask = df['title'].str.contains(r'[가-힣]', na=False)
korean_channel_mask = df['channel_title'].str.contains(r'[가-힣]', na=False)
ko_lang_mask = df['default_language'] == 'ko'
df = df[ko_lang_mask | korean_title_mask | korean_channel_mask]
print("한국 컨텐츠 필터 후:", df.shape)

# 4. 필수값 결측 제거
df = df.dropna(subset=['video_id', 'channel_id', 'collection_date',
                        'published_at', 'category_id', 'view_count'])
print("결측 제거 후:", df.shape)

# 5. 비정상 수치 제거
df = df[df['view_count'] > 0]
df = df[df['rank'].between(1, 200)]
print("비정상 수치 제거 후:", df.shape)

# 6. comment_count 결측 → 0
df['comment_count'] = df['comment_count'].fillna(0)

# 7. 불필요 컬럼 제거
df = df.drop(columns=['description', 'default_language',
                       'default_audio_language', 'live_broadcast_content',
                       'region_code'])

print("\n최종:", df.shape)
print(df.isnull().sum())

중복 제거 후: (872191, 15)
한국 컨텐츠 필터 후: (820630, 15)
결측 제거 후: (820630, 15)
비정상 수치 제거 후: (820630, 15)

최종: (820630, 10)
collection_date    0
rank               0
video_id           0
title              0
published_at       0
channel_id         0
channel_title      0
category_id        0
view_count         0
comment_count      0
dtype: int64


- 사용 칼럼: 'collection_date', 'rank','video_id', 'title','published_at', 'channel_title','category_id','default_language','live_broadcast_content', 'view_count', 'comment_count'

In [8]:
# step2. 카테고리 정규화 (5그룹)

category_map = {
    # Entertainment
    24: 'Entertainment',  # Entertainment
    17: 'Entertainment',  # Sports
    20: 'Entertainment',  # Gaming
    1:  'Entertainment',  # Film & Animation
    23: 'Entertainment',  # Comedy
    # Music
    10: 'Music',
    # Lifestyle
    19: 'Lifestyle',      # Travel & Events
    15: 'Lifestyle',      # Pets & Animals
    2:  'Lifestyle',      # Autos
    22: 'Lifestyle',      # People & Blogs
    26: 'Lifestyle',      # Howto & Style
    # Education
    27: 'Education',
    28: 'Education',      # Science & Tech
    # News
    25: 'News',           # News & Politics
    29: 'News',           # Nonprofits
}

df['category_group'] = df['category_id'].map(category_map).fillna('Other')
print(df['category_group'].value_counts())

category_group
Entertainment    438399
Lifestyle        265241
Music             82645
News              17386
Education         16959
Name: count, dtype: int64


- 주의 포인트: 
- News(2.2%)와 Education(2.0%)이 꽤 적음
- 모델 학습 시 클래스 불균형 문제가 생길 수 있음
- 나중에 모델링 단계에서 class_weight='balanced' 로 처리

In [9]:
# step3. 재진입 이벤트 분리 (6시간 기준)

df = df.sort_values(['video_id', 'collection_date'])

# 스냅샷 간 시간 간격
df['time_gap_h'] = (
    df.groupby('video_id')['collection_date']
      .diff()
      .dt.total_seconds() / 3600
)

# 6시간 이상 공백 → 새 이벤트
df['new_event'] = (df['time_gap_h'] > 6) | (df['time_gap_h'].isna())
df['event_id'] = df.groupby('video_id')['new_event'].cumsum().astype(int)

# 확인
print(df[['video_id', 'collection_date', 'time_gap_h', 'event_id']].head(20))
print(df['event_id'].value_counts().head())

# video_id별 이벤트 수 분포
event_counts = df.groupby('video_id')['event_id'].max()
print(event_counts.value_counts())

           video_id     collection_date  time_gap_h  event_id
395024  --24T6TzQvc 2023-11-11 15:00:00         NaN         1
395223  --24T6TzQvc 2023-11-11 21:00:00         6.0         1
396458  --24T6TzQvc 2023-11-13 09:00:00        36.0         2
396675  --24T6TzQvc 2023-11-13 15:00:00         6.0         2
396877  --24T6TzQvc 2023-11-13 21:00:00         6.0         2
397077  --24T6TzQvc 2023-11-14 03:00:00         6.0         2
397277  --24T6TzQvc 2023-11-14 09:00:00         6.0         2
397485  --24T6TzQvc 2023-11-14 15:00:00         6.0         2
397690  --24T6TzQvc 2023-11-14 21:00:00         6.0         2
397888  --24T6TzQvc 2023-11-15 03:00:00         6.0         2
398097  --24T6TzQvc 2023-11-15 09:00:00         6.0         2
398302  --24T6TzQvc 2023-11-15 15:00:00         6.0         2
398502  --24T6TzQvc 2023-11-15 21:00:00         6.0         2
398702  --24T6TzQvc 2023-11-16 03:00:00         6.0         2
398908  --24T6TzQvc 2023-11-16 09:00:00         6.0         2
399119  

In [10]:
# step4 view_growth_24h 추출 (event_id 기준)

df = df.sort_values(['video_id', 'event_id', 'collection_date'])

# 여기서 T0_time merge 1번
T0_time = (
    df.groupby(['video_id', 'event_id'])['collection_date']
    .min()
    .rename('T0_time')
    .reset_index()
)
if 'T0_time' in df.columns:
    df = df.drop(columns=['T0_time'])

df = df.merge(T0_time, on=['video_id', 'event_id'], how='left')  # ← 1번

T0_view = (
    df.sort_values(['video_id', 'event_id', 'collection_date'])
    .groupby(['video_id', 'event_id'])['view_count']
    .first()
    .rename('T0_view')
    .reset_index()
)

df_24h = df[df['collection_date'] <= df['T0_time'] + pd.Timedelta(hours=24)]
view_at_24h = (
    df_24h.groupby(['video_id', 'event_id'])['view_count']
    .last()
    .rename('view_at_24h')
    .reset_index()
)

view_growth = T0_view.merge(view_at_24h, on=['video_id', 'event_id'], how='left')
view_growth['view_growth_24h'] = (
    view_growth['view_at_24h'] - view_growth['T0_view']
).clip(lower=0)

print(view_growth['view_growth_24h'].describe())

total_events = df.groupby(['video_id', 'event_id']).ngroups
missing = total_events - len(view_growth)
print(f"\n전체 이벤트 수      : {total_events:,}")
print(f"view_growth 행 수   : {len(view_growth):,}")
print(f"누락 이벤트         : {missing:,}")

count          32429.0
mean     139431.975269
std      579342.613685
min                0.0
25%            10220.0
50%            40590.0
75%           111774.0
max         28500012.0
Name: view_growth_24h, dtype: Float64

전체 이벤트 수      : 32,429
view_growth 행 수   : 32,429
누락 이벤트         : 0


In [11]:
print(f"전체 고유 video_id 수 : {df['video_id'].nunique()}")
print(f"view_growth 행 수    : {len(view_growth)}")

# 혹시 24h 데이터가 없는 video_id 있는지 확인
missing_24h = set(df['video_id'].unique()) - set(view_growth['video_id'])
print(f"view_growth 누락 video_id: {len(missing_24h)}개")

전체 고유 video_id 수 : 16739
view_growth 행 수    : 32429
view_growth 누락 video_id: 0개


In [12]:
# step5. 이벤트 집계

# Int64 → float 변환
df['view_count']    = df['view_count'].astype(float)
df['comment_count'] = df['comment_count'].astype(float)

events = df.groupby(['video_id', 'event_id']).agg(
    T0            = ('collection_date', 'min'),
    T_end         = ('collection_date', 'max'),
    channel_id    = ('channel_id', 'first'),
    channel_title = ('channel_title', 'first'),
    category_id   = ('category_id', 'first'),
    category_group= ('category_group', 'first'),
    title         = ('title', 'first'),
    published_at  = ('published_at', 'first'),
    entry_rank    = ('rank', 'first'),      # 진입 당시 순위
    best_rank     = ('rank', 'min'),        # 최고 순위
    peak_view     = ('view_count', 'max'),  # 분석용만 (예측 피처 X)
    T0_view       = ('view_count', 'first'),# 진입 당시 조회수
    T_end_view    = ('view_count', 'last'), # 종료 당시 조회수
    T0_comment    = ('comment_count', 'first'),  # ← 이 줄 추가
    T_end_comment = ('comment_count', 'last'),
).reset_index()

# trending_duration_h (실제 시간차)
events['trending_duration_h'] = (
    events['T_end'] - events['T0']
).dt.total_seconds() / 3600

# 최소 6h 보정 (당일 진입/종료 케이스)
# 짧은 기간 이벤트도 실제 트렌딩 진입 케이스로 확인
events['trending_duration_h'] = events['trending_duration_h'].clip(lower=6)

print(events.shape)
print(events['trending_duration_h'].describe())

(32429, 18)
count    32429.000000
mean       146.272596
std        128.738616
min          6.000000
25%         30.000000
50%        114.000000
75%        240.000000
max        768.000000
Name: trending_duration_h, dtype: float64


In [13]:
# step6. 파생 피처 계산 

import numpy as np
import re

# latency_to_trend_h
events['latency_to_trend_h'] = (
    events['T0'] - events['published_at']
).dt.total_seconds() / 3600
events = events[events['latency_to_trend_h'] >= 0]

# Step 6 수정 — JOIN 키를 video_id → (video_id, event_id)로 변경
events = events.merge(
    view_growth[['video_id', 'event_id', 'view_growth_24h']],
    on=['video_id', 'event_id'], how='left'
)

# ✅ 전체 수치 컬럼 float 변환 (Int64 충돌 방지)
for col in ['T0_view', 'T_end_view', 'T_end_comment', 
            'view_growth_24h', 'entry_rank', 'best_rank']:
    events[col] = events[col].astype(float)

# view_velocity (T_end_view 기준)
events['view_velocity'] = (
    (events['T_end_view'] - events['T0_view'])
    / events['trending_duration_h'].replace(0, np.nan)
).clip(lower=0)

# engagement_ratio
# 기존에 T_end_comment는 트렌딩 끝난 시점 댓글수, 트렌딩이 끝나야 알 수 있는 값이므로
# target leakage가 되고, T0시점(진입 당시) 댓글 수로 바꿔야함. --> T0_comment
events['engagement_ratio'] = (
    events['T0_comment'] / events['T0_view'].replace(0, np.nan)
)

# 시간(hour)을 그대로 쓰면 순환 구조를 못 잡기 때문에,
# sin/cos 변환으로 “원형 시간 구조”를 모델에 알려주는 것
# published_weekday / hour_sin / hour_cos
events['published_weekday'] = events['published_at'].dt.dayofweek
hour = events['published_at'].dt.hour
events['hour_sin'] = np.sin(2 * np.pi * hour / 24)
events['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# has_korean_title
def has_korean(text):
    if pd.isna(text): return 0
    return int(bool(re.search(r'[가-힣]', str(text))))
events['has_korean_title'] = events['title'].apply(has_korean)

# entry_rank_log
events['entry_rank_log'] = np.log1p(events['entry_rank'])

# Winsorization
def winsorize(series, lower=0.05, upper=0.95):
    q_low  = series.quantile(lower)
    q_high = series.quantile(upper)
    return series.clip(q_low, q_high)

events['trending_duration_h_wins'] = winsorize(events['trending_duration_h'])
events['view_velocity_wins']       = winsorize(events['view_velocity'])
events['engagement_ratio_wins']    = winsorize(events['engagement_ratio'])
events['view_count_wins']          = winsorize(events['T_end_view'])
events['view_growth_24h_wins']     = winsorize(events['view_growth_24h'])

print(events.shape)
print(events[['latency_to_trend_h', 'view_velocity',
              'engagement_ratio', 'view_growth_24h']].describe())
print(events.isnull().sum())

(32317, 32)
       latency_to_trend_h  view_velocity  engagement_ratio  view_growth_24h
count        32317.000000   32317.000000      32317.000000     3.231700e+04
mean           108.430456    3876.716603          0.002838     1.380078e+05
std            128.943631   15203.463201          0.003913     5.747511e+05
min              0.009722       0.000000          0.000000     0.000000e+00
25%             15.992222     299.382883          0.000999     1.018700e+04
50%             54.999722     923.822222          0.001773     4.033700e+04
75%            148.478333    2900.555556          0.003212     1.111390e+05
max            837.089722  492185.572464          0.302909     2.850001e+07
video_id                    0
event_id                    0
T0                          0
T_end                       0
channel_id                  0
channel_title               0
category_id                 0
category_group              0
title                       0
published_at                0
entr

In [14]:
# step7. TDI 산출 및 tdi_label 생성

# cat_q95 분모 (이상치 방지)
cat_q95 = events.groupby('category_group')['trending_duration_h'].quantile(0.95)
events['cat_q95_duration'] = events['category_group'].map(cat_q95)

events['TDI'] = (
    (events['trending_duration_h'] / events['cat_q95_duration']).clip(0, 1)
    * (1 - (events['best_rank'] - 1) / 200)
).clip(0, 1)

print(events['TDI'].describe())

# 임계값별 분포 확인
for threshold in [0.3, 0.4, 0.5]:
    pos = (events['TDI'] >= threshold).mean()
    print(f"threshold {threshold} → 1(지속): {pos:.1%} / 0(단명): {1-pos:.1%}")

count    32317.000000
mean         0.315185
std          0.281645
min          0.000070
25%          0.060000
50%          0.248750
75%          0.507042
max          1.000000
Name: TDI, dtype: float64
threshold 0.3 → 1(지속): 45.2% / 0(단명): 54.8%
threshold 0.4 → 1(지속): 35.0% / 0(단명): 65.0%
threshold 0.5 → 1(지속): 25.6% / 0(단명): 74.4%


팀에서 결정해주세요:

"지속 영상을 넓게 정의" → 0.3
"확실히 오래간 영상만" → 0.4~0.5

In [15]:
threshold = 0.4  # 팀 확정 후 수정

events['tdi_label'] = (events['TDI'] >= threshold).astype(int)
print(events['tdi_label'].value_counts())
print(events['tdi_label'].value_counts(normalize=True))

tdi_label
0    21008
1    11309
Name: count, dtype: int64
tdi_label
0    0.65006
1    0.34994
Name: proportion, dtype: float64


In [16]:
# step8. saturation_index 계산

events['T0_date'] = events['T0'].dt.date

cat_sat = (events.groupby(['T0_date', 'category_group'])['video_id']
                 .nunique()
                 .reset_index(name='video_count'))

cat_sat = cat_sat.sort_values(['category_group', 'T0_date'])
cat_sat['saturation_index'] = (
    cat_sat.groupby('category_group')['video_count']
           .transform(lambda x: x / x.rolling(30, min_periods=1).max())
)

avg_dur = (events.groupby(['T0_date', 'category_group'])['trending_duration_h']
                 .mean()
                 .reset_index(name='avg_duration_h'))
cat_sat = cat_sat.merge(avg_dur, on=['T0_date', 'category_group'], how='left')

events = events.merge(
    cat_sat[['T0_date', 'category_group', 'saturation_index']],
    on=['T0_date', 'category_group'], how='left'
)

print(events['saturation_index'].describe())
print(events['saturation_index'].isnull().sum())

count    32317.000000
mean         0.531732
std          0.284191
min          0.008696
25%          0.285714
50%          0.516129
75%          0.750000
max          1.000000
Name: saturation_index, dtype: float64
0


In [17]:
# step9. 최종 저장

feature_cols = [
    # 키
    'video_id', 'event_id', 'channel_id', 'channel_title', 'title',
    # 시간 정보
    'T0', 'T_end', 'published_at', 'T0_date',
    # 타겟 Y
    'trending_duration_h',
    'trending_duration_h_wins',
    'TDI',
    'tdi_label',
    # 피처 X
    'category_group', 'category_id',
    'entry_rank', 'entry_rank_log', 'best_rank',
    'view_growth_24h', 'view_growth_24h_wins',
    'view_velocity', 'view_velocity_wins',
    'engagement_ratio', 'engagement_ratio_wins',
    'saturation_index',
    'latency_to_trend_h',
    'published_weekday', 'hour_sin', 'hour_cos',
    'has_korean_title',
    'view_count_wins',
    # 'cluster_id'  # 클러스터링 후 추가
]

# 누락 컬럼 확인
missing = [c for c in feature_cols if c not in events.columns]
print(f"누락 컬럼: {missing}")

# 저장
video_trending_events = events[feature_cols].copy()
video_trending_events.to_parquet('video_trending_events.parquet', index=False)

# trending_snapshots (원본)
trending_snapshots = df[[
    'video_id', 'collection_date', 'rank',
    'view_count', 'comment_count', 'category_id',
    'channel_id', 'published_at', 'title'
]].copy()
trending_snapshots.to_parquet('trending_snapshots.parquet', index=False)

# category_saturation
cat_sat.to_parquet('category_saturation.parquet', index=False)

print(f"\nvideo_trending_events : {video_trending_events.shape}")
print(f"trending_snapshots    : {trending_snapshots.shape}")
print(f"category_saturation  : {cat_sat.shape}")
print(f"\n결측 확인:\n{video_trending_events.isnull().sum()}")

누락 컬럼: []

video_trending_events : (32317, 31)
trending_snapshots    : (820630, 9)
category_saturation  : (4119, 5)

결측 확인:
video_id                    0
event_id                    0
channel_id                  0
channel_title               0
title                       0
T0                          0
T_end                       0
published_at                0
T0_date                     0
trending_duration_h         0
trending_duration_h_wins    0
TDI                         0
tdi_label                   0
category_group              0
category_id                 0
entry_rank                  0
entry_rank_log              0
best_rank                   0
view_growth_24h             0
view_growth_24h_wins        0
view_velocity               0
view_velocity_wins          0
engagement_ratio            0
engagement_ratio_wins       0
saturation_index            0
latency_to_trend_h          0
published_weekday           0
hour_sin                    0
hour_cos                    0
has_ko

# 전처리 최종 흐름 요약

- 원본 872,191개
- > → STEP 1 : KST 변환 + 한국 크리에이터 필터 + 무결성 처리
- > → STEP 2 : 카테고리 정규화 (5그룹)
- > → STEP 3 : view_growth_24h 추출 (원본 df 기준 T0+24h)
- > → STEP 4 : 재진입 이벤트 분리 (6시간 기준)
- > → STEP 5 : 이벤트 집계 → 32,429개 이벤트
- > → STEP 6 : 파생 피처 + Winsorization
- > → STEP 7 : TDI 산출 + tdi_label
- > → STEP 8 : saturation_index JOIN
- > → STEP 9 : 3개 테이블 저장
- > → 최종 32,317개 재진입 이벤트 (31개 피처)

In [18]:
print(events['trending_duration_h'].describe())
print(events['trending_duration_h'].quantile([0.25, 0.5, 0.75, 0.9, 0.95]))

count    32317.000000
mean       146.366433
std        128.762352
min          6.000000
25%         30.000000
50%        114.000000
75%        240.000000
max        768.000000
Name: trending_duration_h, dtype: float64
0.25     30.0
0.50    114.0
0.75    240.0
0.90    330.0
0.95    378.0
Name: trending_duration_h, dtype: float64


# 임계값 정의 참고

In [19]:
# 확인 코드
events.groupby('tdi_label')['trending_duration_h'].describe()

,count,mean,std,min,25%,50%,75%,max
tdi_label,,,,,,,,
0,21008.0,66.421839,61.628703,6.0,12.0,48.0,102.0,276.0
1,11309.0,294.874348,79.973825,108.0,234.0,282.0,336.0,768.0


In [20]:
for threshold in [0.3, 0.4, 0.5]:
    events['tdi_label_temp'] = (events['TDI'] >= threshold).astype(int)
    print(f"\n=== threshold {threshold} ===")
    print(events.groupby('tdi_label_temp')['trending_duration_h'].describe())
    print(events['tdi_label_temp'].value_counts(normalize=True))


=== threshold 0.3 ===
                  count        mean        std   min    25%    50%    75%  \
tdi_label_temp                                                              
0               17706.0   46.685191  42.884117   6.0   12.0   36.0   72.0   
1               14611.0  267.162823  88.371579  78.0  204.0  258.0  318.0   

                  max  
tdi_label_temp         
0               270.0  
1               768.0  
tdi_label_temp
0    0.547885
1    0.452115
Name: proportion, dtype: float64

=== threshold 0.4 ===
                  count        mean        std    min    25%    50%    75%  \
tdi_label_temp                                                               
0               21008.0   66.421839  61.628703    6.0   12.0   48.0  102.0   
1               11309.0  294.874348  79.973825  108.0  234.0  282.0  336.0   

                  max  
tdi_label_temp         
0               276.0  
1               768.0  
tdi_label_temp
0    0.65006
1    0.34994
Name: proportion, dtype